# 01 - Global Counterfactual Influence Graph Discovery

This notebook selects a discovery cohort, proposes at most four semantic regions with Grad-CAM++, runs raw CCI diffusion interventions progressively by region cardinality, and freezes one graph per target. Re-running cells resumes completed artifacts.

In [ ]:
from pathlib import Path
from argparse import Namespace
import json, logging, os, runpy, subprocess, sys, time
os.environ['PYTHONUNBUFFERED'] = '1'
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)

# The launcher replaces GIT_REF with the exact pushed commit.
REPO_URL = 'https://github.com/lokissdo/cci-diff.git'
GIT_REF = 'main'
PROJECT_ROOT = Path('/kaggle/working/cci-diff')
ASSET_ROOT = Path('/kaggle/input/cci-assets')
DIFFUSION_ASSET_ROOT = Path('/kaggle/input/cci-sd2-assets')
DATA_ROOT = Path('/kaggle/input/celebamask-hq/CelebAMask-HQ')
MODEL_PATH = DIFFUSION_ASSET_ROOT / 'stable-diffusion-2-1'
CLASSIFIER_PATH = ASSET_ROOT / 'resnet50_multilabel_model.pth'
IDENTITY_MODEL_PATH = ASSET_ROOT / 'facenet_vggface2.ts'
IMAGE_ROOT = DATA_ROOT / 'CelebA-HQ-img'
MASK_ROOT = DATA_ROOT / 'CelebAMask-HQ-mask-anno'
OUTPUT_ROOT = Path('/kaggle/working/cci_graph_discovery')

DEVICE = 'cuda'
SAMPLE_COUNT = 300
MAX_SELECTED_REGIONS = 4
SALIENCY_COVERAGE_THRESHOLD = 0.80
SALIENCY_COHORT_FREQUENCY = 0.90
STOP_FLIP_RATE = 0.96
NUM_INFERENCE_STEPS = 35
SEED = 42

TASKS = {
    'smile': {
        'template': PROJECT_ROOT / 'examples/graphs/remove_smile_clean_cci.json',
        'candidate_regions': ['skin', 'nose', 'mouth', 'upper_lip', 'lower_lip', 'left_eye', 'right_eye', 'left_brow', 'right_brow'],
    },
}
DISCOVERY_IDS_PATH = OUTPUT_ROOT / 'discovery_ids.json'

In [ ]:
if not (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', GIT_REF], cwd=PROJECT_ROOT, check=True)
subprocess.run(['git', 'checkout', '--force', '--detach', 'FETCH_HEAD'], cwd=PROJECT_ROOT, check=True)
resolved_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
runtime_packages = ['diffusers', 'transformers', 'accelerate', 'safetensors', 'open-clip-torch', 'grad-cam']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *runtime_packages], check=True)
if not IMAGE_ROOT.is_dir():
    image_roots = sorted(Path('/kaggle/input').rglob('CelebA-HQ-img'))
    if len(image_roots) != 1:
        raise FileNotFoundError(f'Expected one CelebA-HQ-img directory, found: {image_roots}')
    IMAGE_ROOT = image_roots[0]
if not MASK_ROOT.is_dir():
    mask_roots = sorted(Path('/kaggle/input').rglob('CelebAMask-HQ-mask-anno'))
    if len(mask_roots) != 1:
        raise FileNotFoundError(f'Expected one CelebAMask-HQ-mask-anno directory, found: {mask_roots}')
    MASK_ROOT = mask_roots[0]
required = [MODEL_PATH, CLASSIFIER_PATH, IDENTITY_MODEL_PATH, IMAGE_ROOT, MASK_ROOT]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Update the configuration paths; missing: ' + ', '.join(missing))
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT), '--no-deps'], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('CUDA device configured:', DEVICE, 'source commit:', resolved_commit)

In [ ]:
# Select and freeze discovery IDs once. Existing IDs are always reused.
if DISCOVERY_IDS_PATH.is_file():
    discovery_ids = json.loads(DISCOVERY_IDS_PATH.read_text())
else:
    import torch
    from cci_diff.classifiers.celeba_resnet50 import load_celeba_resnet50
    from cci_diff.identity.facenet import build_face_detector
    from scripts.run_clean_cci_pilot import select_eligible_samples

    classifier = load_celeba_resnet50(str(CLASSIFIER_PATH), device=DEVICE, dtype=torch.float32)
    detector = build_face_detector()
    discovery_ids = {}
    selection_args = Namespace(
        max_image_id=30000, image_root=str(IMAGE_ROOT), mask_root=str(MASK_ROOT),
        classifier_input_size=512, device=DEVICE, limit=SAMPLE_COUNT,
        excluded_ids_by_feature={},
    )
    for feature in TASKS:
        selected, _ = select_eligible_samples(selection_args, feature=feature, classifier=classifier, detector=detector)
        discovery_ids[feature] = [sample_id for sample_id, _, _ in selected]
    DISCOVERY_IDS_PATH.write_text(json.dumps(discovery_ids, indent=2))
print({task: len(ids) for task, ids in discovery_ids.items()})

In [ ]:
def run_script(relative_path, arguments):
    script = PROJECT_ROOT / relative_path
    previous_argv = sys.argv[:]
    sys.argv = [str(script), *[str(value) for value in arguments]]
    print(f'[{time.strftime("%H:%M:%S")}] START {relative_path}', flush=True)
    print(' '.join(sys.argv), flush=True)
    try:
        try:
            runpy.run_path(str(script), run_name='__main__')
        except SystemExit as error:
            if error.code not in (None, 0):
                raise
    finally:
        sys.argv = previous_argv
    print(f'[{time.strftime("%H:%M:%S")}] DONE  {relative_path}', flush=True)

for task, config in TASKS.items():
    task_root = OUTPUT_ROOT / task
    screen_dir = task_root / 'screening'
    intervention_dir = task_root / 'interventions'
    graph_dir = task_root / 'graph'
    ids = discovery_ids[task]

    run_script('scripts/screen_counterfactual_regions.py', [
        '--template_graph', config['template'], '--classifier_path', CLASSIFIER_PATH,
        '--sample_ids', *ids, '--candidate_regions', *config['candidate_regions'],
        '--max_selected_regions', MAX_SELECTED_REGIONS,
        '--saliency_coverage_threshold', SALIENCY_COVERAGE_THRESHOLD,
        '--cohort_frequency_threshold', SALIENCY_COHORT_FREQUENCY,
        '--minimum_captured_saliency', 0.0,
        '--image_root', IMAGE_ROOT, '--mask_root', MASK_ROOT,
        '--device', DEVICE, '--output_dir', screen_dir,
    ])
    screen = json.loads((screen_dir / 'screening_manifest.json').read_text())
    candidates = screen['selected_candidate_regions'][:MAX_SELECTED_REGIONS]

    run_script('scripts/run_counterfactual_region_interventions.py', [
        '--template_graph', config['template'], '--sample_ids', *ids,
        '--candidate_regions', *candidates, '--max_set_size', len(candidates),
        '--stop_flip_rate', STOP_FLIP_RATE, '--seeds', SEED,
        '--image_root', IMAGE_ROOT, '--mask_root', MASK_ROOT,
        '--model_path', MODEL_PATH, '--classifier_path', CLASSIFIER_PATH,
        '--identity_model_path', IDENTITY_MODEL_PATH,
        '--num_inference_steps', NUM_INFERENCE_STEPS,
        '--device', DEVICE, '--torch_dtype', 'auto',
        '--python_executable', sys.executable, '--continue_on_error',
        '--output_dir', intervention_dir,
    ])

    run_script('scripts/discover_counterfactual_graph.py', [
        '--results', intervention_dir / 'intervention_results.csv',
        '--template_graph', config['template'], '--required_flip_rate', STOP_FLIP_RATE,
        '--minimum_samples', SAMPLE_COUNT, '--bootstrap_samples', 2000,
        '--confidence', 0.95, '--random_seed', SEED, '--output_dir', graph_dir,
    ])

In [ ]:
import pandas as pd
for task in TASKS:
    print('\n', task.upper())
    display(pd.read_csv(OUTPUT_ROOT / task / 'graph' / 'region_set_metrics.csv').sort_values(['flip_rate', 'mean_effect'], ascending=False).head(15))